In [71]:
# ============================================================
# RENABAP · Análisis 1 — Universo y evolución (AMT)
# BLOQUE 1: Carga + armonización + geometría
# Entorno: Google Colab | Fuente de datos: repo GitHub Camilamop/RENABAP
# ============================================================
import os
REPO_URL = "https://github.com/Camilamop/RENABAP.git"

if os.path.exists('/content/RENABAP'):
    !cd /content/RENABAP && git fetch origin && git reset --hard origin/master
else:
    !git clone {REPO_URL} /content/RENABAP

!pip install geopandas --quiet

# chequeo rápido: ¿el 2023 ya trae WKT?
import pandas as pd
cols = pd.read_csv('/content/RENABAP/data/raw/RENABAP_2023.csv', nrows=0).columns.tolist()
print("¿WKT en 2023?", "WKT" in cols, "| columnas:", len(cols))

HEAD is now at 79737c0 Merge branch 'master' of https://github.com/Camilamop/RENABAP
¿WKT en 2023? True | columnas: 19


In [72]:
!pip install geopandas --quiet

In [73]:

import pandas as pd
import geopandas as gpd
from shapely import wkt as shapely_wkt

RAIZ = Path("/content/RENABAP")
DIR_DATOS  = RAIZ / "data" / "raw"
DIR_SALIDA = RAIZ / "data" / "processed"
DIR_SALIDA.mkdir(parents=True, exist_ok=True)

CRS_ORIGEN = "EPSG:4326"



In [74]:
# ============================================================
# Diccionario de armonización + esquema núcleo + ARCHIVOS
# nombre_original -> nombre_núcleo
# ============================================================
MAPEO = {
    2018: {
        "id_renabap": "id_renabap",
        "Nombre del barrio": "nombre_barrio",
        "Nombre de provincia": "provincia",
        "Nombre de departamentos/comuna": "departamento",
        "Localidad": "localidad",
        "Cantidad de familias": "cant_familias",
        "Tamaño (km2)": "superficie_km2",
        "Año de creación": "anio_creacion",
        "WKT": "wkt",
    },
    2022: {
        "renabap_id": "id_renabap",
        "nombre_barrio": "nombre_barrio",
        "provincia": "provincia",
        "departamento": "departamento",
        "localidad": "localidad",
        "cantidad_familias_aproximada": "cant_familias",
        "superficie_m2": "superficie_m2",
        "decada_de_creacion": "decada_creacion",
        "WKT": "wkt",
    },
    2023: {
        "id_renabap": "id_renabap",
        "nombre_barrio": "nombre_barrio",
        "provincia": "provincia",
        "departamento": "departamento",
        "localidad": "localidad",
        "cantidad_familias_aproximada": "cant_familias",
        "superficie_m2": "superficie_m2",
        "decada_de_creacion": "decada_creacion",
        "WKT": "wkt",
    },
}

NUCLEO = [
    "id_renabap", "registro", "nombre_barrio", "provincia", "departamento",
    "localidad", "cant_familias", "superficie_km2",
    "anio_creacion", "decada_creacion", "wkt",
]

ARCHIVOS = {
    2018: DIR_DATOS / "RENABAP_2018.csv",
    2022: DIR_DATOS / "RENABAP_2022.csv",
    2023: DIR_DATOS / "RENABAP_2023.csv",
}

# chequeo temprano: avisar si falta algún archivo
for anio, ruta in ARCHIVOS.items():
    print(f"  {anio}: {'OK' if ruta.exists() else 'FALTA -> ' + str(ruta)}")



  2018: OK
  2022: OK
  2023: OK


In [75]:
# ============================================================
# Funciones de carga y armonización
# ============================================================
def cargar_crudo(anio: int, ruta: Path) -> pd.DataFrame:
    df = pd.read_csv(ruta, dtype=str, keep_default_na=False, encoding="utf-8")
    df.columns = [c.strip() for c in df.columns]   # limpia espacios en headers
    return df


def normalizar_decada(serie_cruda: pd.Series) -> pd.Series:
    """Extrae el año de textos tipo 'Década 1990' y lo deja como entero (1990)."""
    s = serie_cruda.astype(str).str.extract(r"(\d{4})")[0]
    return pd.to_numeric(s, errors="coerce").astype("Int64")


def armonizar(anio: int, df: pd.DataFrame) -> pd.DataFrame:
    mapa = MAPEO[anio]

    cols_presentes = {orig: nuevo for orig, nuevo in mapa.items() if orig in df.columns}
    out = df[list(cols_presentes.keys())].rename(columns=cols_presentes).copy()

    # registro (edición del registro)
    out["registro"] = anio

    # id como texto limpio
    out["id_renabap"] = out["id_renabap"].str.strip()

    # superficie a km2 (2022 y 2023 vienen en m2)
    if "superficie_m2" in out.columns:
        out["superficie_km2"] = pd.to_numeric(out["superficie_m2"], errors="coerce") / 1_000_000
        out = out.drop(columns=["superficie_m2"])
    elif "superficie_km2" in out.columns:
        out["superficie_km2"] = pd.to_numeric(out["superficie_km2"], errors="coerce")

    # cantidad de familias -> numérico
    if "cant_familias" in out.columns:
        out["cant_familias"] = pd.to_numeric(out["cant_familias"], errors="coerce").astype("Int64")

    # año de creación -> numérico (solo 2018 lo trae confiable)
    if "anio_creacion" in out.columns:
        anio_num = pd.to_numeric(out["anio_creacion"], errors="coerce")
        anio_num = anio_num.where((anio_num >= 1800) & (anio_num <= 2025))  # descarta 0 y basura
        out["anio_creacion"] = anio_num.astype("Int64")
    else:
        out["anio_creacion"] = pd.array([pd.NA] * len(out), dtype="Int64")


    if "decada_creacion" in out.columns:
        out["decada_creacion"] = normalizar_decada(out["decada_creacion"])
    else:
        out["decada_creacion"] = (out["anio_creacion"] // 10 * 10).astype("Int64")

    # localidad: garantizar la columna aunque alguna edición no la trajera
    if "localidad" not in out.columns:
        out["localidad"] = pd.NA

    # reordenar a esquema núcleo (agrega faltantes como NA, p.ej. 'wkt' en 2023)
    for c in NUCLEO:
        if c not in out.columns:
            out[c] = pd.NA
    return out[NUCLEO]


def a_geodataframe(df: pd.DataFrame) -> gpd.GeoDataFrame:
    """Parsea el WKT a geometría y arma un GeoDataFrame en EPSG:4326."""
    geom = df["wkt"].apply(lambda w: shapely_wkt.loads(w) if isinstance(w, str) and w.strip() else None)
    gdf = gpd.GeoDataFrame(df.drop(columns=["wkt"]), geometry=geom, crs=CRS_ORIGEN)
    return gdf


In [76]:
# ============================================================
# Ejecución
# ============================================================
def main():
    piezas = []
    print("Cargando y armonizando los tres registros...\n")
    for anio, ruta in ARCHIVOS.items():
        crudo = cargar_crudo(anio, ruta)
        arm = armonizar(anio, crudo)
        piezas.append(arm)
        n_geom = arm["wkt"].apply(lambda w: isinstance(w, str) and w.strip() != "").sum()
        print(f"  {anio}: {len(arm):>5} barrios | geometrías no vacías: {n_geom}")

    maestra = pd.concat(piezas, ignore_index=True)
    gdf = a_geodataframe(maestra)
    invalidas = gdf.geometry.isna().sum()

    print(f"\nTabla maestra (largo): {len(maestra)} filas "
          f"{maestra['registro'].value_counts().sort_index().to_dict()}")
    print(f"Geometrías que no parsearon o faltan: {invalidas}")

    # CSV de atributos (sin geometría)
    maestra.drop(columns=["wkt"]).to_csv(DIR_SALIDA / "renabap_maestra_largo.csv", index=False)

    # FIX: escritura robusta del GPKG por capas (sin pisar capas ni romper con capa vacía)
    gpkg_path = DIR_SALIDA / "renabap_nacional.gpkg"
    if gpkg_path.exists():
        gpkg_path.unlink()
    primera = True
    for anio in ARCHIVOS:
        sub = gdf[(gdf["registro"] == anio) & gdf.geometry.notna()]
        if len(sub) == 0:
            print(f"  [AVISO] {anio}: sin geometrías -> no se escribe capa en el GPKG.")
            continue
        sub.to_file(gpkg_path, layer=f"renabap_{anio}", driver="GPKG",
                    mode="w" if primera else "a")
        primera = False

    print(f"\nExportado en: {DIR_SALIDA.resolve()}")
    print("  - renabap_maestra_largo.csv")
    print("  - renabap_nacional.gpkg (una capa por edición con geometría disponible)")

    print("\n--- Familias totales por registro (nacional) ---")
    print(maestra.groupby("registro")["cant_familias"].agg(["count", "sum"]))
    print("\n--- Cobertura de 'década de creación' por registro ---")
    print(maestra.groupby("registro")["decada_creacion"].apply(lambda s: round(s.notna().mean(), 3)))
    print("\n--- Cobertura de 'localidad' por registro ---")
    print(maestra.groupby("registro")["localidad"].apply(
        lambda s: round((s.notna() & (s.astype(str).str.strip() != "")).mean(), 3)))

    # aviso explícito si algún año quedó sin geometría (hoy: 2023)
    sin_geo = [a for a in ARCHIVOS if gdf[(gdf["registro"] == a) & gdf.geometry.notna()].empty]
    if sin_geo:
        print("\n" + "="*60)
        print(f"⚠️  SIN GEOMETRÍA: {sin_geo}. Para el recorte AMT y los mapas")
        print("    necesitás la versión con polígonos de esa(s) edición(es)")
        print("    (GeoJSON/SHP/GPKG oficial o CSV con columna WKT) en data/raw/.")
        print("="*60)

    return maestra, gdf


maestra, gdf = main()


Cargando y armonizando los tres registros...

  2018:  4416 barrios | geometrías no vacías: 4416
  2022:  5687 barrios | geometrías no vacías: 5687
  2023:  6467 barrios | geometrías no vacías: 6467

Tabla maestra (largo): 16570 filas {2018: 4416, 2022: 5687, 2023: 6467}
Geometrías que no parsearon o faltan: 0

Exportado en: /content/RENABAP/data/processed
  - renabap_maestra_largo.csv
  - renabap_nacional.gpkg (una capa por edición con geometría disponible)

--- Familias totales por registro (nacional) ---
          count      sum
registro                
2018       4416   925609
2022       5687  1165275
2023       6467  1237795

--- Cobertura de 'década de creación' por registro ---
registro
2018    0.679
2022    1.000
2023    1.000
Name: decada_creacion, dtype: float64

--- Cobertura de 'localidad' por registro ---
registro
2018    1.0
2022    1.0
2023    1.0
Name: localidad, dtype: float64
